<a href="https://colab.research.google.com/github/DogwonLee/Causal-Inference---Rent/blob/main/Causal_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Step2

In [ ]:
step2_cleaning_segmentation.ipynb

텍스트 셀 <undefined>
# %% [markdown]
# 2단계 — 텍스트 정제 및 기사 분리 (Cleaning & Article Segmentation)

**과제:** Causal Inference and AI 기말 프로젝트 (Track B)
**주제:** 1920년 6월 워싱턴 D.C. *Evening Star* 신문에 실린 **임대료위원회(Rent Commission)** 판결 기사
**대상 일자:** 1920-06-02, 1920-06-10, 1920-06-12, 1920-06-19, 1920-06-20

이 노트북은 6단계 파이프라인 중 **2단계(정제·정규화)**를 수행합니다. 1단계(소스 선정)는 이미 끝난 상태이고, 위 5개 일자의 OCR 텍스트 파일을 입력으로 사용합니다.

### 이번 단계에서 하는 일
1. OCR 텍스트에서 페이지 구분 마커 제거
2. 줄바꿈으로 끊긴 단어 복원 (예: `depriv-` + `ed` → `deprived`)
3. 헤드라인(대문자 위주 줄)과 본문을 구분해 "기사" 단위로 분리
4. 임대료위원회 관련 키워드로 1차 후보 기사 골라내기 (정밀 선별은 3단계에서)

**선택한 옵션:** 정제 단계 Option C (정규식·규칙 기반 정규화) — 외부 라이브러리 없이 Python 표준 라이브러리만 사용 (Colab 환경에서 인터넷 연결 없이도 동작)


텍스트 셀 <undefined>
# %% [markdown]
## 0. 데이터 파일 업로드

아래 5개의 `.txt` 파일을 업로드하세요 (Library of Congress *Chronicling America* 아카이브, Evening Star 1920년 6월):

- `1920-06-02.txt`
- `1920-06-10.txt`
- `1920-06-12.txt`
- `1920-06-19.txt`
- `1920-06-20.txt`

업로드 버튼을 누르고 파일 선택 창에서 5개를 한 번에 선택하면 됩니다.

코드 셀 <undefined>
# %% [code]
import sys
import os

IN_COLAB = "google.colab" in sys.modules

DATA_DIR = "data"
OUTPUT_DIR = "output"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if IN_COLAB:
    from google.colab import files
    print("5개 파일을 선택하세요: 1920-06-02 / 06-10 / 06-12 / 06-19 / 06-20 (.txt)")
    uploaded = files.upload()
    for fname in uploaded:
        target = os.path.join(DATA_DIR, fname)
        os.replace(fname, target)
else:
    print(f"Colab 환경이 아닙니다. '{DATA_DIR}/' 폴더에 5개 파일을 직접 넣고 다음 셀로 진행하세요.")

EXPECTED = [
    "1920-06-02.txt", "1920-06-10.txt", "1920-06-12.txt",
    "1920-06-19.txt", "1920-06-20.txt",
]
present = os.listdir(DATA_DIR)
print("\n현재 data/ 폴더 내용:", present)
missing = [f for f in EXPECTED if f not in present]
if missing:
    print("아직 없는 파일:", missing)
else:
    print("5개 파일 모두 확인됨")


텍스트 셀 <undefined>
# %% [markdown]
## 1. 정제 함수 정의

신문 OCR 텍스트는 페이지마다 `===== PAGE N =====` 같은 구분자가 있고, 헤드라인은 거의 항상 대문자로 인쇄되어 있습니다. 아래 함수들은 이 구조를 이용해 텍스트를 "기사" 단위로 잘라냅니다.

- `split_pages` — 페이지 구분자를 기준으로 텍스트를 나눕니다.
- `is_headline_line` — 한 줄이 헤드라인인지 판단합니다 (길이 3~90자, 대문자 비율 85% 이상).
- `join_body_lines` — OCR 줄바꿈을 하나의 문단으로 합치고, 줄 끝의 하이픈(-)을 복원합니다.
- `segment_articles` — 헤드라인 블록 다음에 오는 본문 블록을 하나의 "기사"로 묶습니다.

코드 셀 <undefined>
# %% [code]
import re

PAGE_MARKER_RE = re.compile(r"^=====.*=====$")


def load_raw(path):
    with open(path, encoding="utf-8", errors="replace") as f:
        return f.read()


def split_pages(raw):
    """'===== PAGE N =====' 마커를 기준으로 (페이지번호, 줄 리스트) 목록을 반환."""
    lines = raw.replace("\r\n", "\n").split("\n")
    pages = []
    current_page_num = 0
    current_lines = []
    for line in lines:
        m = re.match(r"=====\s*PAGE\s*(\d+)\s*=====", line.strip())
        if m:
            if current_lines:
                pages.append((current_page_num, current_lines))
            current_page_num = int(m.group(1))
            current_lines = []
            continue
        if PAGE_MARKER_RE.match(line.strip()):
            continue  # 맨 위 "===== Evening Star - DATE =====" 배너
        current_lines.append(line)
    if current_lines:
        pages.append((current_page_num, current_lines))
    return pages


def is_headline_line(line):
    """짧고(3~90자) 대문자 비율이 높은 줄을 헤드라인으로 간주."""
    s = line.strip()
    if not (3 <= len(s) <= 90):
        return False
    letters = [c for c in s if c.isalpha()]
    if len(letters) < 3:
        return False
    upper_ratio = sum(c.isupper() for c in letters) / len(letters)
    return upper_ratio > 0.85


def join_body_lines(lines):
    """OCR 줄들을 하나의 문단으로 합치고, 줄 끝 하이픈을 복원."""
    out = ""
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if out.endswith("-"):
            out = out[:-1] + s
        elif out:
            out = out + " " + s
        else:
            out = s
    return out


def segment_articles(pages):
    """헤드라인 블록 + 그 다음 본문 블록을 하나의 기사로 묶어 리스트로 반환."""
    articles = []
    for page_num, lines in pages:
        headline_buf = []
        body_buf = []

        def flush():
            if headline_buf or body_buf:
                articles.append({
                    "page": page_num,
                    "headline": join_body_lines(headline_buf),
                    "body": join_body_lines(body_buf),
                })

        for line in lines:
            if is_headline_line(line):
                if body_buf:
                    flush()
                    headline_buf, body_buf = [], []
                headline_buf.append(line)
            else:
                body_buf.append(line)
        flush()
    return [a for a in articles if len(a["headline"]) + len(a["body"]) > 15]


텍스트 셀 <undefined>
# %% [markdown]
## 2. 임대료위원회 관련 기사 1차 필터링

5개 일자에는 정치 대회, 스포츠, 부고, 광고 등 수많은 기사가 섞여 있습니다. 아래 키워드 중 하나라도 포함된 기사를 "1차 후보"로 골라냅니다.

**주의:** 이 필터는 정밀하지 않습니다 — 부동산 광고나 무관한 기사도 일부 섞여 들어옵니다. 진짜 임대료위원회 "사건" 기사만 골라내는 정밀 작업은 3단계(정보 추출)에서 진행합니다.

코드 셀 <undefined>
# %% [code]
import json
import glob

KEYWORDS = [
    "rent commission", "rent law", "ball act", "ball rent",
    "petition", "determination", "increase rent", "reduce rent",
    "fair rent", "rents", "landlord", "tenant", "possession of",
]


def is_relevant(article):
    text = (article["headline"] + " " + article["body"]).lower()
    return any(kw in text for kw in KEYWORDS)


summary = {}

for path in sorted(glob.glob(os.path.join(DATA_DIR, "1920-06-*.txt"))):
    date = os.path.basename(path).replace(".txt", "")
    raw = load_raw(path)
    pages = split_pages(raw)
    articles = segment_articles(pages)
    relevant = [a for a in articles if is_relevant(a)]

    with open(os.path.join(OUTPUT_DIR, f"{date}_articles.json"), "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)
    with open(os.path.join(OUTPUT_DIR, f"{date}_relevant.json"), "w", encoding="utf-8") as f:
        json.dump(relevant, f, ensure_ascii=False, indent=2)

    summary[date] = {
        "total_articles": len(articles),
        "relevant_candidates": len(relevant),
    }

with open(os.path.join(OUTPUT_DIR, "clean_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

for date, s in summary.items():
    print(f"{date}: 전체 기사 {s['total_articles']}건 중 후보 {s['relevant_candidates']}건")


텍스트 셀 <undefined>
# %% [markdown]
## 3. 결과 미리보기

각 일자에서 "본문이 100자 이상이고 'rent commission'을 직접 언급하는" 기사를 몇 개 출력해, 실제 위원회 판결 기사가 잘 잡혔는지 확인합니다.

코드 셀 <undefined>
# %% [code]
for path in sorted(glob.glob(os.path.join(OUTPUT_DIR, "1920-06-*_relevant.json"))):
    date = os.path.basename(path).replace("_relevant.json", "")
    data = json.load(open(path, encoding="utf-8"))
    shown = 0
    print(f"\n===== {date} =====")
    for a in data:
        if "rent commission" in a["body"].lower() and len(a["body"]) > 100:
            print(f"[HEADLINE] {a['headline'][:80]}")
            print(f"[BODY]     {a['body'][:300]}...")
            print()
            shown += 1
        if shown >= 3:
            break
    if shown == 0:
        print("(이 날짜에는 'rent commission' 직접 언급 기사가 없음)")


텍스트 셀 <undefined>
# %% [markdown]
## 정리

- `output/{date}_articles.json` — 해당 일자에서 분리된 전체 기사 목록
- `output/{date}_relevant.json` — 임대료위원회 관련 키워드를 포함한 1차 후보 기사 목록
- `output/clean_summary.json` — 일자별 기사 수 요약

다음 **3단계(정보 추출)**에서는 이 `*_relevant.json` 파일들을 입력으로 받아, 실제 임대료위원회 "사건" 기사만 골라내고 각 사건에서 청원인 · 피청원인 · 부동산 주소 · 청원 유형 · 판정 결과를 구조화된 형태로 추출합니다.

### 다운로드 (선택)
아래 셀을 실행하면 `output/` 폴더를 zip으로 묶어 다운로드할 수 있습니다.

코드 셀 <undefined>
# %% [code]
import shutil

zip_path = "step2_output"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"{zip_path}.zip 생성 완료")

if IN_COLAB:
    from google.colab import files
    files.download(f"{zip_path}.zip")


